In [1]:
import os
import sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
spark = get_spark_session()
print("Project root:", PROJECT_ROOT)

26/09/24 12:32:52 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/09/24 12:32:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 12:32:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Project root: /Users/thilini/Documents/MSc/Data_intensive_computing/Assignments/Urban_Mobility_Data_Platform


26/09/24 12:32:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from src.integration.integrated_taxi_trips import build_integrated_taxi_trips_incremental, GOLD_PATH
from src.common.delta_writer import write_delta

new_gold_df, n_new = build_integrated_taxi_trips_incremental(spark)

if new_gold_df is None:
    print("Gold: no new trips to add, skipping rebuild")
else:
    write_delta(
        new_gold_df,
        GOLD_PATH,
        mode="append",
        partition_by=["pickup_year", "pickup_month"],
        schema_mode="merge",
    )
    print(f"Gold: appended {n_new} new trip rows incrementally")

/Users/thilini/.pyenv/versions/3.12.0/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
26/09/24 12:33:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Gold: appended 293670 new trip rows incrementally
